# Rossmann Store Sales — Feature Engineering per RNN

Notebook di preparazione dati per un modello ricorrente (RNN) di forecasting delle vendite. Carichiamo train / validation / test, uniamo le informazioni dei negozi (`store.csv`) e applichiamo la pipeline di feature engineering richiesta: encoding ciclico di data/giorno della settimana, encoding delle variabili categoriche, gestione dei valori mancanti e trasformazione del target.

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "dataset"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


In [2]:
print("=" * 70)
print("1. CARICAMENTO DATI: train, validation, test + merge con store")
print("=" * 70)

# StateHoliday viene letta come stringa per evitare che pandas mescoli
# il valore 0 (int) con "a"/"b"/"c" (str) nella stessa colonna (colonna a
# tipo misto -> warning e comportamento inconsistente nelle operazioni
# successive, es. il confronto "!= '0'" usato piu' avanti).
train_raw = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv"),
    parse_dates=["Date"], dtype={"StateHoliday": str}, low_memory=False,
)
test_raw = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv"),
    parse_dates=["Date"], dtype={"StateHoliday": str}, low_memory=False,
)
store = pd.read_csv(os.path.join(DATA_DIR, "store.csv"))

print(f"train.csv: {train_raw.shape[0]:,} righe, {train_raw.shape[1]} colonne")
print(f"test.csv:  {test_raw.shape[0]:,} righe, {test_raw.shape[1]} colonne")
print(f"store.csv: {store.shape[0]:,} righe, {store.shape[1]} colonne")

# Merge con le informazioni sul negozio (StoreType, Assortment, concorrenza,
# Promo2, ...), disponibili sia per il train che per il test
train_full = train_raw.merge(store, how="left", on="Store")
test_df = test_raw.merge(store, how="left", on="Store")

print(f"\ntrain unito a store: {train_full.shape[0]:,} righe, {train_full.shape[1]} colonne")
print(f"test unito a store:  {test_df.shape[0]:,} righe, {test_df.shape[1]} colonne")


1. CARICAMENTO DATI: train, validation, test + merge con store


train.csv: 1,017,209 righe, 9 colonne
test.csv:  41,088 righe, 8 colonne
store.csv: 1,115 righe, 10 colonne

train unito a store: 1,017,209 righe, 18 colonne
test unito a store:  41,088 righe, 17 colonne


In [3]:
print("\n" + "=" * 70)
print("2. SPLIT TEMPORALE: train / validation")
print("=" * 70)

# Non disponiamo di un file di validation separato: lo ricaviamo dal train,
# tenendo da parte le ultime settimane in ordine CRONOLOGICO (mai uno split
# casuale su una serie storica: mescolare le date causerebbe data leakage,
# facendo "vedere" al modello in training giorni futuri rispetto a quelli di
# validazione). La finestra di validation replica la durata del test
# ufficiale (~6 settimane), cosi' la metrica calcolata in validazione e'
# rappresentativa di quella attesa sul test.
VAL_WEEKS = 6

train_full = train_full.sort_values("Date").reset_index(drop=True)
cutoff_date = train_full["Date"].max() - pd.Timedelta(weeks=VAL_WEEKS)

train_df = train_full[train_full["Date"] <= cutoff_date].copy()
val_df = train_full[train_full["Date"] > cutoff_date].copy()

print(f"Train:      {len(train_df):>8,} righe  ({train_df['Date'].min().date()} -> {train_df['Date'].max().date()})")
print(f"Validation: {len(val_df):>8,} righe  ({val_df['Date'].min().date()} -> {val_df['Date'].max().date()})")
print(f"Test:       {len(test_df):>8,} righe  ({test_df['Date'].min().date()} -> {test_df['Date'].max().date()})")



2. SPLIT TEMPORALE: train / validation


Train:       970,379 righe  (2013-01-01 -> 2015-06-19)
Validation:   46,830 righe  (2015-06-20 -> 2015-07-31)
Test:         41,088 righe  (2015-08-01 -> 2015-09-17)


In [4]:
print("\n" + "=" * 70)
print("3. CORREZIONE VALORI MANCANTI NOTI NEL TEST UFFICIALE")
print("=" * 70)

# Bug noto del test set ufficiale Rossmann: la colonna Open ha alcuni valori
# mancanti per lo Store 622. Li impostiamo a 1 (negozio aperto): e' l'ipotesi
# piu' ragionevole, dato che negli altri giorni della stessa settimana quello
# store risulta regolarmente aperto, e comunque necessaria per non avere NaN
# in input alla rete.
n_missing_open = test_df["Open"].isnull().sum()
print(f"Valori mancanti in Open (test): {n_missing_open}")
test_df["Open"] = test_df["Open"].fillna(1).astype(int)



3. CORREZIONE VALORI MANCANTI NOTI NEL TEST UFFICIALE
Valori mancanti in Open (test): 11


In [5]:
print("\n" + "=" * 70)
print("4. STATISTICHE CALCOLATE SOLO SUL TRAINING SET")
print("=" * 70)

# La mediana di CompetitionDistance viene calcolata SOLO sul training set e
# poi riusata (senza ricalcolarla) per riempire i mancanti anche in
# validation e test: se la calcolassimo sull'intero dataset, faremmo
# trapelare nel training un'informazione statistica derivata anche da
# osservazioni future (data leakage).
competition_distance_median = train_df["CompetitionDistance"].median()
print(f"Mediana CompetitionDistance (dal training set): {competition_distance_median}")



4. STATISTICHE CALCOLATE SOLO SUL TRAINING SET
Mediana CompetitionDistance (dal training set): 2330.0


## Pipeline di feature engineering

Definiamo un'unica funzione `engineer_features` e la applichiamo identica a train, validation e test: questo garantisce che le tre tabelle finiscano con le **stesse colonne, nello stesso ordine e con la stessa logica di trasformazione**, requisito indispensabile per alimentare correttamente una rete neurale. Le uniche informazioni "esterne" alla singola riga (la mediana di `CompetitionDistance`) vengono passate come parametro, calcolate una sola volta sul training set (cella precedente).

In [6]:
def engineer_features(
    df,
    competition_distance_median,
    is_train,
    drop_store=True,
    dayofweek_sincos=True,
    date_sincos=True,
    drop_customers=True,
    stateholiday_bool=True,
    storetype_onehot=True,
    assortment_int=True,
    fill_competition_distance=True,
    competition_open_months=True,
    promo2_open_weeks=True,
    promo_interval_bool=True,
    sales_log=True,
):
    """
    Applica al dataframe (train, validation oppure test) le trasformazioni
    di feature engineering per il modello RNN. Ogni trasformazione puo'
    essere attivata/disattivata singolarmente tramite i parametri booleani,
    per poter sperimentare facilmente con sottoinsiemi diversi di feature.

    Parametri
    ---------
    df : dataframe grezzo (train/val gia' unito a store, oppure test unito
         a store)
    competition_distance_median : mediana di CompetitionDistance calcolata
         SOLO sul training set, riusata per train/val/test
    is_train : True per train/validation (contengono la colonna Sales),
         False per il test ufficiale (che non la contiene)
    drop_store : rimuove la colonna Store (identificativo negozio)
    dayofweek_sincos : trasforma DayOfWeek in DayOfWeek_sin/DayOfWeek_cos
         e rimuove la colonna originale
    date_sincos : ricava DayOfYear_sin/DayOfYear_cos (stagionalita' annuale,
         periodo 365 giorni) e rimuove la colonna Date
    drop_customers : rimuove la colonna Customers (non nota a tempo di
         predizione)
    stateholiday_bool : converte StateHoliday da stringa ("0","a","b","c")
         a booleano/intero 0/1 (0 = nessuna festivita')
    storetype_onehot : converte StoreType in one-hot encoding
    assortment_int : converte Assortment in intero (a=0, b=1, c=2)
    fill_competition_distance : riempie i valori mancanti di
         CompetitionDistance con la mediana passata come parametro
    competition_open_months : sostituisce CompetitionOpenSinceMonth/Year con
         CompetitionOpenMonths ("da quanti mesi e' aperta la concorrenza"),
         mancanti e negativi impostati a 0
    promo2_open_weeks : sostituisce Promo2SinceYear/Week con Promo2OpenWeeks
         ("da quante settimane e' attiva Promo2"), mancanti e negativi
         impostati a 0
    promo_interval_bool : sostituisce PromoInterval con il booleano
         IsPromoMonth (il mese corrente e' un mese di ripartenza di Promo2?)
    sales_log : aggiunge la colonna SalesLog = log(1 + Sales) (solo se
         is_train=True e la colonna Sales e' presente)
    """
    df = df.copy()

    # Variabili temporanee derivate dalla data: servono a piu' trasformazioni
    # (feature cicliche, mesi/settimane di apertura di concorrenza/Promo2,
    # mese per IsPromoMonth), quindi le calcoliamo comunque se la colonna
    # Date e' ancora presente, indipendentemente da quali flag sono attivi.
    if "Date" in df.columns:
        _year = df["Date"].dt.year
        _month = df["Date"].dt.month
        _day_of_year = df["Date"].dt.dayofyear
        _week_of_year = df["Date"].dt.isocalendar().week.astype(int)

    # ------------------------------------------------------------------
    # Giorno dell'anno -> seno/coseno (stagionalita' annuale: dicembre e
    # gennaio sono "vicini" nel tempo, cosa che un singolo intero 1..365 non
    # rappresenta correttamente). Rimuove poi la colonna Date.
    # ------------------------------------------------------------------
    if date_sincos:
        df["DayOfYear_sin"] = np.sin(2 * np.pi * _day_of_year / 365)
        df["DayOfYear_cos"] = np.cos(2 * np.pi * _day_of_year / 365)
        df = df.drop(columns=["Date"])

    # ------------------------------------------------------------------
    # Giorno della settimana -> seno/coseno (stesso ragionamento sul ciclo
    # settimanale). Rimuove poi la colonna DayOfWeek originale.
    # ------------------------------------------------------------------
    if dayofweek_sincos:
        df["DayOfWeek_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
        df["DayOfWeek_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)
        df = df.drop(columns=["DayOfWeek"])

    # ------------------------------------------------------------------
    # Customers: rimossa perche' non e' nota a tempo di predizione (non e'
    # presente nel test set ufficiale ed e' fortemente correlata a Sales:
    # usarla in training sarebbe una forma di leakage)
    # ------------------------------------------------------------------
    if drop_customers and "Customers" in df.columns:
        df = df.drop(columns=["Customers"])

    # ------------------------------------------------------------------
    # StateHoliday: da stringa ("0","a","b","c") a booleano/intero 0/1
    # 0 -> nessuna festivita', a/b/c -> festivita' (di qualunque tipo)
    # ------------------------------------------------------------------
    if stateholiday_bool:
        df["StateHoliday"] = (df["StateHoliday"] != "0").astype(int)

    # ------------------------------------------------------------------
    # StoreType: one-hot encoding (4 categorie: a, b, c, d). Non essendo una
    # variabile ordinale, il one-hot evita di introdurre un ordinamento
    # arbitrario tra i tipi di negozio, a differenza di un semplice intero.
    # ------------------------------------------------------------------
    if storetype_onehot:
        storetype_dummies = pd.get_dummies(df["StoreType"], prefix="StoreType").astype(int)
        df = pd.concat([df.drop(columns=["StoreType"]), storetype_dummies], axis=1)

    # ------------------------------------------------------------------
    # Assortment: label encoding esplicito e ordinato a=0, b=1, c=2
    # ------------------------------------------------------------------
    if assortment_int:
        df["Assortment"] = df["Assortment"].map({"a": 0, "b": 1, "c": 2})

    # ------------------------------------------------------------------
    # CompetitionDistance: valori mancanti (negozi senza un concorrente
    # mappato) riempiti con la mediana calcolata sul training set
    # ------------------------------------------------------------------
    if fill_competition_distance:
        df["CompetitionDistance"] = df["CompetitionDistance"].fillna(competition_distance_median)

    # ------------------------------------------------------------------
    # CompetitionOpenSinceMonth/Year -> "da quanti mesi e' aperta la
    # concorrenza" rispetto alla data della riga corrente. Mancanti e
    # negativi -> 0 ("nessuna concorrenza attiva al momento").
    # ------------------------------------------------------------------
    if competition_open_months:
        competition_open_months_val = (
            12 * (_year - df["CompetitionOpenSinceYear"])
            + (_month - df["CompetitionOpenSinceMonth"])
        )
        competition_open_months_val = competition_open_months_val.fillna(0).clip(lower=0)
        df["CompetitionOpenMonths"] = competition_open_months_val
        df = df.drop(columns=["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"])

    # ------------------------------------------------------------------
    # Promo2SinceYear/Week -> stesso trattamento, analogo a quello della
    # concorrenza: "da quante settimane e' attiva Promo2" per quel negozio.
    # Mancanti e negativi -> 0.
    # ------------------------------------------------------------------
    if promo2_open_weeks:
        promo2_open_weeks_val = (
            52 * (_year - df["Promo2SinceYear"])
            + (_week_of_year - df["Promo2SinceWeek"])
        )
        promo2_open_weeks_val = promo2_open_weeks_val.fillna(0).clip(lower=0)
        df["Promo2OpenWeeks"] = promo2_open_weeks_val
        df = df.drop(columns=["Promo2SinceWeek", "Promo2SinceYear"])

    # ------------------------------------------------------------------
    # PromoInterval -> booleano IsPromoMonth: il mese della riga corrente
    # rientra tra quelli in cui riparte un ciclo Promo2 per quel negozio?
    # (PromoInterval elenca i mesi abbreviati in inglese, es. "Jan,Apr,Jul,Oct")
    # ------------------------------------------------------------------
    if promo_interval_bool:
        month_abbr = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                      7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
        current_month_str = _month.map(month_abbr)
        promo_interval_list = df["PromoInterval"].fillna("").str.split(",")
        df["IsPromoMonth"] = [
            int(m in lst) for m, lst in zip(current_month_str, promo_interval_list)
        ]
        df = df.drop(columns=["PromoInterval"])

    # ------------------------------------------------------------------
    # Store: rimossa se richiesto (l'identificativo negozio non viene usato
    # come feature nel modello)
    # ------------------------------------------------------------------
    if drop_store:
        df = df.drop(columns=["Store"])

    # ------------------------------------------------------------------
    # Target in scala logaritmica: log(1 + Sales). Il test ufficiale non
    # contiene Sales (e' cio' che va predetto), quindi si applica solo a
    # train/validation.
    # ------------------------------------------------------------------
    if sales_log and is_train and "Sales" in df.columns:
        df["SalesLog"] = np.log1p(df["Sales"])

    return df


In [7]:
print("\n" + "=" * 70)
print("5. APPLICAZIONE DELLA PIPELINE A TRAIN / VALIDATION / TEST")
print("=" * 70)

# Applichiamo la STESSA funzione, con la STESSA mediana (calcolata sul solo
# training set), ai tre dataframe: questo garantisce coerenza tra le feature
# di train, validation e test ed evita data leakage.
train_prep = engineer_features(train_df, competition_distance_median, is_train=True)
val_prep = engineer_features(val_df, competition_distance_median, is_train=True)
test_prep = engineer_features(test_df, competition_distance_median, is_train=False)

print(f"train_prep: {train_prep.shape[0]:,} righe, {train_prep.shape[1]} colonne")
print(f"val_prep:   {val_prep.shape[0]:,} righe, {val_prep.shape[1]} colonne")
print(f"test_prep:  {test_prep.shape[0]:,} righe, {test_prep.shape[1]} colonne")

print("\nColonne train_prep:")
print(list(train_prep.columns))
print("\nColonne test_prep (nessun target, presente Id per la submission):")
print(list(test_prep.columns))



5. APPLICAZIONE DELLA PIPELINE A TRAIN / VALIDATION / TEST


train_prep: 970,379 righe, 20 colonne
val_prep:   46,830 righe, 20 colonne
test_prep:  41,088 righe, 19 colonne

Colonne train_prep:
['Sales', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Assortment', 'CompetitionDistance', 'Promo2', 'DayOfYear_sin', 'DayOfYear_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'CompetitionOpenMonths', 'Promo2OpenWeeks', 'IsPromoMonth', 'SalesLog']

Colonne test_prep (nessun target, presente Id per la submission):
['Id', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Assortment', 'CompetitionDistance', 'Promo2', 'DayOfYear_sin', 'DayOfYear_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d', 'CompetitionOpenMonths', 'Promo2OpenWeeks', 'IsPromoMonth']


In [8]:
print("\n" + "=" * 70)
print("6. CONTROLLI DI QUALITA'")
print("=" * 70)

# Nessun valore mancante deve restare dopo la pipeline (requisito per dare
# in input i dati a una rete neurale)
print("Valori mancanti - train:", train_prep.isnull().sum().sum())
print("Valori mancanti - val:  ", val_prep.isnull().sum().sum())
print("Valori mancanti - test: ", test_prep.isnull().sum().sum())

print("\nTipi di dato (train_prep):")
print(train_prep.dtypes)

print("\nAnteprima (train_prep):")
print(train_prep.head())



6. CONTROLLI DI QUALITA'
Valori mancanti - train: 0
Valori mancanti - val:   0
Valori mancanti - test:  0

Tipi di dato (train_prep):
Sales                      int64
Open                       int64
Promo                      int64
StateHoliday               int64
SchoolHoliday              int64
Assortment                 int64
CompetitionDistance      float64
Promo2                     int64
DayOfYear_sin            float64
DayOfYear_cos            float64
DayOfWeek_sin            float64
DayOfWeek_cos            float64
StoreType_a                int64
StoreType_b                int64
StoreType_c                int64
StoreType_d                int64
CompetitionOpenMonths    float64
Promo2OpenWeeks          float64
IsPromoMonth               int64
SalesLog                 float64
dtype: object

Anteprima (train_prep):
   Sales  Open  Promo  StateHoliday  SchoolHoliday  Assortment  CompetitionDistance  Promo2  DayOfYear_sin  DayOfYear_cos  DayOfWeek_sin  \
0      0     0      0     

In [9]:
print("\n" + "=" * 70)
print("7. SALVATAGGIO DATASET PREPARATI")
print("=" * 70)

train_prep.to_csv(os.path.join(OUTPUT_DIR, "train_prepared.csv"), index=False)
val_prep.to_csv(os.path.join(OUTPUT_DIR, "val_prepared.csv"), index=False)
test_prep.to_csv(os.path.join(OUTPUT_DIR, "test_prepared.csv"), index=False)

print(f"File salvati in '{OUTPUT_DIR}/':")
print("  - train_prepared.csv")
print("  - val_prepared.csv")
print("  - test_prepared.csv  (contiene 'Id', da riusare nella submission)")

print("\n" + "=" * 70)
print("FEATURE ENGINEERING COMPLETATO")
print("=" * 70)



7. SALVATAGGIO DATASET PREPARATI


File salvati in 'outputs/':
  - train_prepared.csv
  - val_prepared.csv
  - test_prepared.csv  (contiene 'Id', da riusare nella submission)

FEATURE ENGINEERING COMPLETATO
